<a href="https://colab.research.google.com/github/rogernogueira/caam/blob/main/Grid_PiPE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [104]:
!pip install catboost


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 5.1 MB/s eta 0:00:00


In [105]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import xgboost
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import catboost



In [2]:
df  = pd.read_csv('https://gist.githubusercontent.com/rogernogueira/30a55552e6b64f89ab09b552eb93fb05/raw/600f1a01a8cf4c10b8330b4d48ba663f71fae143/heart.csv')

In [37]:
df.isna().sum()

,0
ge,0
Sex,0
ChestPainType,0
RestingBP,0
Cholesterol,0
FastingBS,0
RestingECG,0
MaxHR,0
ExerciseAngina,0
Oldpeak,0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ge              918 non-null    int64  
 1   Sex             918 non-null    object 
 2   ChestPainType   918 non-null    object 
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    object 
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    object 
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    object 
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 86.2+ KB


In [13]:
pd.get_dummies(df, columns =['Sex', 'ChestPainType','RestingECG', 'ST_Slope', 'ExerciseAngina'], drop_first=True  )

,ge,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_Normal,RestingECG_ST,ST_Slope_Flat,ST_Slope_Up,ExerciseAngina_Y
0,40,140,289,0,172,0.0,0,True,True,False,False,True,False,False,True,False
1,49,160,180,0,156,1.0,1,False,False,True,False,True,False,True,False,False
2,37,130,283,0,98,0.0,0,True,True,False,False,False,True,False,True,False
3,48,138,214,0,108,1.5,1,False,False,False,False,True,False,True,False,True
4,54,150,195,0,122,0.0,0,True,False,True,False,True,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,110,264,0,132,1.2,1,True,False,False,True,True,False,True,False,False
914,68,144,193,1,141,3.4,1,True,False,False,False,True,False,True,False,False
915,57,130,131,0,115,1.2,1,True,False,False,False,True,False,True,False,True
916,57,130,236,0,174,0.0,1,False,True,False,False,False,False,True,False,False


In [17]:
df['ChestPainType'].unique()

array(['ATA', 'NAP', 'ASY', 'TA'], dtype=object)

In [19]:
cat_col = ['Sex', 'ChestPainType','RestingECG', 'ST_Slope', 'ExerciseAngina', 'FastingBS']
num_col = ['ge','RestingBP','Cholesterol', 'MaxHR','Oldpeak']

In [21]:
num_pipe = Pipeline([ ('scaler', StandardScaler())])
cat_pipe = Pipeline([ ('encoder', OneHotEncoder(handle_unknown='ignore'))])

In [28]:
preprocess = ColumnTransformer([('num', num_pipe, num_col),
                                ('cat', cat_pipe, cat_col)])

In [29]:
X = df.drop('HeartDisease', axis = 1)
y = df['HeartDisease']

In [31]:
X_train,X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [30]:
xgb_pipe = Pipeline(
    [
        ('prep', preprocess),
        ('xgb', xgboost.XGBClassifier())
    ]
)

In [32]:
xgb_pipe.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['ge', 'RestingBP',
                                                   'Cholesterol', 'MaxHR',
                                                   'Oldpeak']),
                                                 ('cat',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Sex', 'ChestPainType',
                                                   'RestingECG', 'ST_Slope',
                                                   'ExerciseAngina',
                                                   'FastingBS'])])),
                ('xgb',
                 XGBClassifier(base_score=...
                               feature_types=None, gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=None,
                               num_parallel_tree=None, random_state=None, ...))])

In [34]:
y_pred  = xgb_pipe.predict(X_test)

In [38]:
print('Acuracia: ', accuracy_score(y_test, y_pred))
print('Precisão', precision_score(y_test, y_pred))
print('Recall: ', recall_score(y_test, y_pred))
print('AUC: ',roc_auc_score(y_test, y_pred))

Acuracia:  0.8731884057971014
Precisão 0.910828025477707
Recall:  0.8719512195121951
AUC:  0.8734756097560976


In [45]:
rf_pipe = Pipeline(
    [
        ('prep', preprocess),
        ('rf_cls', RandomForestClassifier())
    ]
)

In [47]:
rf_pipe.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['ge', 'RestingBP',
                                                   'Cholesterol', 'MaxHR',
                                                   'Oldpeak']),
                                                 ('cat',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Sex', 'ChestPainType',
                                                   'RestingECG', 'ST_Slope',
                                                   'ExerciseAngina',
                                                   'FastingBS'])])),
                ('rf_cls', RandomForestClassifier())])

In [48]:
y_pred_rf = rf_pipe.predict(X_test)

In [50]:
print('Acuracia: ', accuracy_score(y_test, y_pred_rf))
print('Precisão', precision_score(y_test,y_pred_rf))
print('Recall: ', recall_score(y_test, y_pred_rf))
print('AUC: ',roc_auc_score(y_test, y_pred_rf))

Acuracia:  0.894927536231884
Precisão 0.9192546583850931
Recall:  0.9024390243902439
AUC:  0.8931837979094076


In [81]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


In [73]:
params_dist = {
    'xgb__n_estimators': [100, 200, 300],
    #'xgb__learning_rate': [0.01, 0.1, 0.2],
    'xgb__max_depth': [3, 4, 5],
    'xgb__gamma':[0,0.1, 0.3,0.5]
}

In [74]:
grid = GridSearchCV(
    estimator=xgb_pipe,
    param_grid=params_dist,
    cv=cv,
    scoring='roc_auc',
    verbose=2,
    n_jobs=-1
)


In [75]:
grid.fit(X_train, y_train)

Fitting 5 folds for each of 36 candidates, totalling 180 fits


GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('prep',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('scaler',
                                                                                          StandardScaler())]),
                                                                         ['ge',
                                                                          'RestingBP',
                                                                          'Cholesterol',
                                                                          'MaxHR',
                                                                          'Oldpeak']),
                                                                        ('cat',
                                                                         Pipeline(steps=[('encoder',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['Sex',
                                                                          'ChestPainType',
                                                                          'Restin...
                                                      max_cat_to_onehot=None,
                                                      max_delta_step=None,
                                                      max_depth=None,
                                                      max_leaves=None,
                                                      min_child_weight=None,
                                                      missing=nan,
                                                      monotone_constraints=None,
                                                      multi_strategy=None,
                                                      n_estimators=None,
                                                      n_jobs=None,
                                                      num_parallel_tree=None,
                                                      random_state=None, ...))]),
             n_jobs=-1,
             param_grid={'xgb__gamma': [0, 0.1, 0.3, 0.5],
                         'xgb__max_depth': [3, 4, 5],
                         'xgb__n_estimators': [100, 200, 300]},
             scoring='roc_auc', verbose=2)

In [76]:
grid.best_params_

{'xgb__gamma': 0.5, 'xgb__max_depth': 4, 'xgb__n_estimators': 100}

In [77]:
best_model = grid.best_estimator_

In [78]:
y_pred = best_model.predict(X_test)

In [79]:
print('Acuracia: ', accuracy_score(y_test, y_pred))
print('Precisão', precision_score(y_test, y_pred))
print('Recall: ', recall_score(y_test, y_pred))
print('AUC: ',roc_auc_score(y_test, y_pred))

Acuracia:  0.8586956521739131
Precisão 0.9032258064516129
Recall:  0.8536585365853658
AUC:  0.8598649825783972


array(['ge', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol',
       'FastingBS', 'RestingECG', 'MaxHR', 'ExerciseAngina', 'Oldpeak',
       'ST_Slope'], dtype=object)

In [100]:
params_dist = {
    'rf_cls__n_estimators': [100, 200, 300, 400, 500, ],
    'rf_cls__bootstrap': [True, False],
    'rf_cls__max_depth': [3, 4, 5, 6, 7,8],

}

In [101]:
grid_rf = GridSearchCV(
    estimator=rf_pipe,
    param_grid=params_dist,
    cv=cv,
    scoring='roc_auc',
    verbose=2,
    n_jobs=-1
)

In [102]:
grid_rf.fit(X_train, y_train)

Fitting 5 folds for each of 60 candidates, totalling 300 fits


GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('prep',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('scaler',
                                                                                          StandardScaler())]),
                                                                         ['ge',
                                                                          'RestingBP',
                                                                          'Cholesterol',
                                                                          'MaxHR',
                                                                          'Oldpeak']),
                                                                        ('cat',
                                                                         Pipeline(steps=[('encoder',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['Sex',
                                                                          'ChestPainType',
                                                                          'RestingECG',
                                                                          'ST_Slope',
                                                                          'ExerciseAngina',
                                                                          'FastingBS'])])),
                                       ('rf_cls', RandomForestClassifier())]),
             n_jobs=-1,
             param_grid={'rf_cls__bootstrap': [True, False],
                         'rf_cls__max_depth': [3, 4, 5, 6, 7, 8],
                         'rf_cls__n_estimators': [100, 200, 300, 400, 500]},
             scoring='roc_auc', verbose=2)

In [89]:
model_rf  = grid_rf.best_estimator_

In [90]:
y_pred = model_rf.predict(X_test)

In [91]:
print('Acuracia: ', accuracy_score(y_test, y_pred))
print('Precisão', precision_score(y_test, y_pred))
print('Recall: ', recall_score(y_test, y_pred))
print('AUC: ',roc_auc_score(y_test, y_pred))

Acuracia:  0.8913043478260869
Precisão 0.9135802469135802
Recall:  0.9024390243902439
AUC:  0.8887195121951219


In [92]:
grid_rf.best_params_

{'rf_cls__max_depth': 5, 'rf_cls__n_estimators': 300}

In [93]:
rf_pipe2 = Pipeline(
    [
        ('prep', preprocess),
        ('rf_cls', RandomForestClassifier(max_depth=5, n_estimators = 300))
    ]
)

In [94]:
rf_pipe2.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['ge', 'RestingBP',
                                                   'Cholesterol', 'MaxHR',
                                                   'Oldpeak']),
                                                 ('cat',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Sex', 'ChestPainType',
                                                   'RestingECG', 'ST_Slope',
                                                   'ExerciseAngina',
                                                   'FastingBS'])])),
                ('rf_cls',
                 RandomForestClassifier(max_depth=5, n_estimators=300))])

In [95]:
y_pred = rf_pipe2.predict(X_test)

In [96]:
print('Acuracia: ', accuracy_score(y_test, y_pred))
print('Precisão', precision_score(y_test, y_pred))
print('Recall: ', recall_score(y_test, y_pred))
print('AUC: ',roc_auc_score(y_test, y_pred))

Acuracia:  0.8840579710144928
Precisão 0.9074074074074074
Recall:  0.8963414634146342
AUC:  0.8812064459930314


In [106]:
cls = catboost.CatBoostClassifier()

In [107]:
catb_piper = Pipeline(
    [
        ('prep', preprocess),
        ('catb', catboost.CatBoostClassifier())
    ]
)

In [108]:
catb_piper.fit(X_train, y_train)

Learning rate set to 0.008526
0:	learn: 0.6862650	total: 51.8ms	remaining: 51.8s
1:	learn: 0.6790617	total: 65.4ms	remaining: 32.7s
2:	learn: 0.6724127	total: 70.3ms	remaining: 23.4s
3:	learn: 0.6670929	total: 75.3ms	remaining: 18.8s
4:	learn: 0.6612325	total: 79.4ms	remaining: 15.8s
5:	learn: 0.6554049	total: 87.4ms	remaining: 14.5s
6:	learn: 0.6487819	total: 95.4ms	remaining: 13.5s
7:	learn: 0.6421386	total: 104ms	remaining: 12.8s
8:	learn: 0.6364015	total: 112ms	remaining: 12.4s
9:	learn: 0.6299049	total: 136ms	remaining: 13.5s
10:	learn: 0.6258432	total: 137ms	remaining: 12.3s
11:	learn: 0.6209787	total: 153ms	remaining: 12.6s
12:	learn: 0.6150979	total: 155ms	remaining: 11.7s
13:	learn: 0.6087922	total: 165ms	remaining: 11.6s
14:	learn: 0.6034734	total: 167ms	remaining: 11s
15:	learn: 0.5990161	total: 171ms	remaining: 10.5s
16:	learn: 0.5934377	total: 175ms	remaining: 10.1s
17:	learn: 0.5871371	total: 177ms	remaining: 9.64s
18:	learn: 0.5813970	total: 181ms	remaining: 9.33s
19:	le

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['ge', 'RestingBP',
                                                   'Cholesterol', 'MaxHR',
                                                   'Oldpeak']),
                                                 ('cat',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Sex', 'ChestPainType',
                                                   'RestingECG', 'ST_Slope',
                                                   'ExerciseAngina',
                                                   'FastingBS'])])),
                ('catb',
                 <catboost.core.CatBoostClassifier object at 0x7e3d3a6248d0>)])

In [109]:
y_pred  = catb_piper.predict(X_test)

In [110]:
print('Acuracia: ', accuracy_score(y_test, y_pred))
print('Precisão', precision_score(y_test, y_pred))
print('Recall: ', recall_score(y_test, y_pred))
print('AUC: ',roc_auc_score(y_test, y_pred))

Acuracia:  0.8985507246376812
Precisão 0.930379746835443
Recall:  0.8963414634146342
AUC:  0.8990635888501743
